# ClipCap fixed test end-to-end trên Google Colab

Notebook này clone repository, đọc feature cache và năm checkpoint Transformer Mapper từ Google Drive, sinh caption cho toàn bộ `fixed_test_round_001`, tính CIDEr, BLEU-4, CLIPScore và RefCLIPScore, rồi lưu một gói kết quả dùng cho báo cáo.

Trước khi chạy, hãy chọn **Runtime > Change runtime type > T4 GPU**. Notebook hỗ trợ resume: nếu Colab bị ngắt, chạy lại với cùng `RUN_TAG` để tiếp tục những ảnh chưa hoàn thành.

## 1. Mount Drive, clone repository và đặt đường dẫn

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/HnhanBk415/zfs-clip-image-captioning.git'
BRANCH = 'refactor/huuthien/evaluation'
PROJECT_ROOT = Path('/content/zfs-clip-image-captioning')

if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH],
        check=True,
    )
elif PROJECT_ROOT.exists():
    raise RuntimeError(
        f'{PROJECT_ROOT} đã tồn tại nhưng không phải Git repository'
    )
else:
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )

DRIVE_ROOT = Path('/content/drive/MyDrive/clipcap_colab')
FEATURE_CACHE_PATH = DRIVE_ROOT / 'data_cache' / 'features' / 'clip_features.pt'
CHECKPOINT_ROOT = DRIVE_ROOT / 'experiments_fixed_epoch'
INFERENCE_OUTPUT_BASE = DRIVE_ROOT / 'evaluation_outputs'
METRICS_OUTPUT_BASE = DRIVE_ROOT / 'metrics'
REPORT_OUTPUT_BASE = DRIVE_ROOT / 'reports'
RUN_TAG = 'fixed_test_round_001_report'

for directory in (
    INFERENCE_OUTPUT_BASE,
    METRICS_OUTPUT_BASE,
    REPORT_OUTPUT_BASE,
):
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Working directory: {Path.cwd()}')
print(f'Git branch: {BRANCH}')
print(f'Feature cache: {FEATURE_CACHE_PATH}')
print(f'Checkpoint root: {CHECKPOINT_ROOT}')
print(f'Run tag: {RUN_TAG}')

## 2. Cài và kiểm tra môi trường

In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
        check=True,
    )

if importlib.util.find_spec('pycocoevalcap') is None:
    raise RuntimeError('Không cài được pycocoevalcap')
if shutil.which('java') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'openjdk-17-jre-headless'],
        check=True,
    )
if shutil.which('java') is None:
    raise RuntimeError('Java không có trong PATH; chưa thể tính CIDEr/BLEU-4')

subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['java', '-version'], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError('Notebook cần Colab GPU để chạy đủ năm checkpoint')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.get_device_name(0)}')

## 3. Kiểm tra fixed test, feature cache và đủ năm checkpoint

In [ ]:
from src.clipcap.inference.checkpoints import (
    REQUIRED_ARTIFACTS,
    validate_artifact_directory,
)
from src.clipcap.inference.features import load_feature_cache
from src.config.clipcap_config import (
    CLIPCAP_DEFAULT_INFERENCE_CONFIG,
    CLIPCAP_TRAIN_SEED,
    CLIPCAP_TRAIN_SUBSETS,
)
from src.config.common_config import SPLIT_DIR

SUBSETS = tuple(CLIPCAP_TRAIN_SUBSETS)
EXPECTED_SUBSETS = (
    'train_1pct',
    'train_5pct',
    'train_10pct',
    'train_25pct',
    'train_100pct',
)
SEED = CLIPCAP_TRAIN_SEED
MANIFEST_PATH = SPLIT_DIR / 'fixed_test_round_001.json'
REFERENCES_PATH = SPLIT_DIR / 'test.json'

if SUBSETS != EXPECTED_SUBSETS:
    raise ValueError(f'Danh sách checkpoint không đúng: {SUBSETS}')
if not FEATURE_CACHE_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy feature cache: {FEATURE_CACHE_PATH}')
for path in (MANIFEST_PATH, REFERENCES_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy split file: {path}')

with MANIFEST_PATH.open('r', encoding='utf-8') as file:
    fixed_test_ids = json.load(file)
with REFERENCES_PATH.open('r', encoding='utf-8') as file:
    test_references = json.load(file)
if len(fixed_test_ids) != 104 or len(set(fixed_test_ids)) != 104:
    raise ValueError('fixed_test_round_001 phải chứa đúng 104 image ID duy nhất')
missing_references = set(fixed_test_ids) - set(test_references)
if missing_references:
    raise ValueError(f'Thiếu reference cho {len(missing_references)} ảnh')
if any(len(test_references[image_id]) != 5 for image_id in fixed_test_ids):
    raise ValueError('Mỗi ảnh fixed test phải có đúng năm reference')

checkpoint_rows = []
for subset_name in SUBSETS:
    checkpoint_dir = CHECKPOINT_ROOT / subset_name / f'seed_{SEED}'
    artifacts = validate_artifact_directory(checkpoint_dir)
    checkpoint_rows.append({
        'subset': subset_name,
        'directory': str(checkpoint_dir),
        'artifacts': len(artifacts),
    })

cached_features = load_feature_cache(FEATURE_CACHE_PATH, fixed_test_ids)
if len(cached_features) != len(fixed_test_ids):
    raise ValueError('Feature cache không phủ đủ fixed test')
del cached_features

import pandas as pd
from IPython.display import display

display(pd.DataFrame(checkpoint_rows))
print(f'Fixed test images: {len(fixed_test_ids)}')
print(f'Checkpoint artifacts required: {list(REQUIRED_ARTIFACTS)}')
print('Tất cả input đã sẵn sàng')

## 4. Chạy inference cho cả năm checkpoint

Không thêm `--no-resume`, vì vậy chạy lại cell này sẽ tiếp tục các ảnh còn thiếu thay vì sinh lại từ đầu.

In [ ]:
INFERENCE_OUTPUT_ROOT = INFERENCE_OUTPUT_BASE / 'test' / RUN_TAG
inference_config = CLIPCAP_DEFAULT_INFERENCE_CONFIG

inference_command = [
    sys.executable,
    '-m',
    'src.clipcap.inference.run_inference',
    '--dataset',
    'fixed_test_round_001',
    '--allow-test',
    '--run-tag',
    RUN_TAG,
    '--split-dir',
    str(SPLIT_DIR),
    '--checkpoint-root',
    str(CHECKPOINT_ROOT),
    '--feature-cache',
    str(FEATURE_CACHE_PATH),
    '--output-base',
    str(INFERENCE_OUTPUT_BASE),
    '--subsets',
    *SUBSETS,
    '--seed',
    str(SEED),
    '--device',
    'cuda',
    '--image-batch-size',
    str(inference_config.image_batch_size),
    '--max-new-tokens',
    str(inference_config.max_new_tokens),
    '--num-beams',
    str(inference_config.num_beams),
    '--num-return-sequences',
    str(inference_config.num_return_sequences),
    '--length-penalty',
    str(inference_config.length_penalty),
]
inference_command.append(
    '--early-stopping'
    if inference_config.early_stopping
    else '--no-early-stopping'
)

print('Bắt đầu inference năm checkpoint...')
subprocess.run(inference_command, cwd=PROJECT_ROOT, check=True)
print(f'Inference hoàn tất: {INFERENCE_OUTPUT_ROOT}')

## 5. Xác minh và lưu năm file caption

In [ ]:
prediction_paths = {}
caption_paths = {}
coverage_rows = []
expected_ids = set(fixed_test_ids)

for subset_name in SUBSETS:
    model_output_dir = INFERENCE_OUTPUT_ROOT / subset_name / f'seed_{SEED}'
    prediction_path = model_output_dir / 'predictions.jsonl'
    if not prediction_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy prediction: {prediction_path}')

    records = []
    with prediction_path.open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f'JSONL lỗi tại {prediction_path}:{line_number}'
                ) from error
            records.append(record)

    prediction_ids = [record.get('image_id') for record in records]
    if len(prediction_ids) != len(set(prediction_ids)):
        raise ValueError(f'{subset_name} chứa image ID trùng')
    missing = expected_ids - set(prediction_ids)
    extra = set(prediction_ids) - expected_ids
    if missing or extra:
        raise ValueError(
            f'{subset_name}: missing={len(missing)}, extra={len(extra)}'
        )

    captions = {record['image_id']: record['caption'] for record in records}
    caption_path = model_output_dir / 'captions.json'
    with caption_path.open('w', encoding='utf-8') as file:
        json.dump(captions, file, ensure_ascii=False, indent=2)

    prediction_paths[subset_name] = prediction_path
    caption_paths[subset_name] = caption_path
    coverage_rows.append({
        'checkpoint': subset_name,
        'captions': len(captions),
        'complete': len(captions) == len(expected_ids),
        'path': str(caption_path),
    })

display(pd.DataFrame(coverage_rows))

## 6. Tính CIDEr, BLEU-4, CLIPScore và RefCLIPScore

In [ ]:
METRICS_OUTPUT_DIR = METRICS_OUTPUT_BASE / 'test' / RUN_TAG
RUN_CONFIG_PATH = INFERENCE_OUTPUT_ROOT / 'run_config.json'
if not RUN_CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy run config: {RUN_CONFIG_PATH}')

metrics_command = [
    sys.executable,
    '-m',
    'src.common.caption_metrics',
    '--inference-manifest',
    str(MANIFEST_PATH),
    '--references',
    str(REFERENCES_PATH),
    '--feature-cache',
    str(FEATURE_CACHE_PATH),
    '--output-dir',
    str(METRICS_OUTPUT_DIR),
    '--run-config',
    str(RUN_CONFIG_PATH),
    '--clip-batch-size',
    '32',
    '--device',
    'cuda',
    '--references-per-image',
    '5',
]
for subset_name in SUBSETS:
    metrics_command.extend([
        '--prediction',
        f'{subset_name}={prediction_paths[subset_name]}',
    ])

print('Bắt đầu tính metric cho năm checkpoint...')
subprocess.run(metrics_command, cwd=PROJECT_ROOT, check=True)
print(f'Metric hoàn tất: {METRICS_OUTPUT_DIR}')

## 7. Đóng gói kết quả dùng cho báo cáo

In [ ]:
REPORT_DIR = REPORT_OUTPUT_BASE / 'fixed_test_round_001' / RUN_TAG
REPORT_CAPTIONS_DIR = REPORT_DIR / 'captions'
REPORT_PREDICTIONS_DIR = REPORT_DIR / 'predictions'
REPORT_METRICS_DIR = REPORT_DIR / 'metrics'
for directory in (
    REPORT_CAPTIONS_DIR,
    REPORT_PREDICTIONS_DIR,
    REPORT_METRICS_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

for subset_name in SUBSETS:
    metric_path = prediction_paths[subset_name].parent / 'metrics.json'
    if not metric_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy metric: {metric_path}')
    shutil.copy2(
        caption_paths[subset_name],
        REPORT_CAPTIONS_DIR / f'{subset_name}.json',
    )
    shutil.copy2(
        prediction_paths[subset_name],
        REPORT_PREDICTIONS_DIR / f'{subset_name}.jsonl',
    )
    shutil.copy2(
        metric_path,
        REPORT_METRICS_DIR / f'{subset_name}.json',
    )

comparison_path = METRICS_OUTPUT_DIR / 'comparison.json'
per_image_path = METRICS_OUTPUT_DIR / 'per_image_comparison.csv'
for source_path in (comparison_path, per_image_path, RUN_CONFIG_PATH):
    if not source_path.is_file():
        raise FileNotFoundError(f'Thiếu output: {source_path}')
    shutil.copy2(source_path, REPORT_DIR / source_path.name)

with comparison_path.open('r', encoding='utf-8') as file:
    comparison = json.load(file)
results_table = pd.DataFrame(comparison['results']).sort_values('rank')
results_table_path = REPORT_DIR / 'results_table.csv'
results_table.to_csv(results_table_path, index=False, encoding='utf-8')

archive_path = shutil.make_archive(
    str(REPORT_DIR),
    'zip',
    root_dir=REPORT_DIR,
)
display(results_table[[
    'rank',
    'experiment',
    'CIDEr',
    'BLEU-4',
    'CLIPScore',
    'RefCLIPScore',
]])
print(f'Report directory: {REPORT_DIR}')
print(f'Report archive: {archive_path}')
print(f'Results table: {results_table_path}')
print('Fixed test inference và evaluation đã hoàn tất')

## Cấu trúc kết quả trên Drive

```text
clipcap_colab/reports/fixed_test_round_001/<RUN_TAG>/
├── captions/                 # 5 file caption gọn
├── predictions/              # 5 file inference chi tiết
├── metrics/                  # 5 file metric riêng
├── comparison.json           # kết quả tổng hợp và metadata
├── per_image_comparison.csv  # metric cho từng ảnh
├── results_table.csv         # bảng đưa vào báo cáo
└── run_config.json           # cấu hình tái lập inference
```

Một file ZIP cùng tên với thư mục report cũng được tạo bên cạnh để lưu trữ hoặc tải xuống.